In [ ]:
import pandas as pd

import torch

from transformers import AutoTokenizer

from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset, DataLoader

from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence



# загрузка датасета
raw = pd.read_csv('yelp_reviews.csv')


texts = raw['text']
labels = raw['label']


# разделение выборки на трейн и тест
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)


# создание токенизатора с помощью класса AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name) # создайте токенизатор


# токенизируем тексты
train_texts_tokenized = tokenizer(train_texts.tolist(), truncation=True)['input_ids']
val_texts_tokenized = tokenizer(val_texts.tolist(), truncation=True)['input_ids']


# создаём класс кастомного, наследуясь от класса Dataset из PyTorch

class YelpDataset(Dataset):
    # в конструкторе просто сохраняем тексты и классы
    def __init__(self, texts, labels, max_len=256):
        self.texts = texts # сохраните тексты
        self.labels = labels # сохраните классы
        self.max_len = max_len # сохрание max_len


    # возвращаем размер датасета (кол-во текстов)
    def __len__(self):
        return len(self.texts) # верните размер датасета
        
    def __getitem__(self, idx):
        # возвращаем текст и его класс
        # для текста ограничиваем длину
        # не делаем никаких доп. преобразований как padding и masking
        return {
            'text': torch.tensor(self.texts[idx][:self.max_len], dtype=torch.long), # верните текст под индексом idx в виде тензора, ограничьте его длиной self.max_len
            'label': torch.tensor(self.labels[idx], dtype=torch.long) # верните класс под индексом idx в виде тензора
        }


# кастомная функция collate_fn для формирования батчей
def collate_fn(batch):
    texts = [torch.tensor(item['text'] for item in batch)] # получите список текстов в батче
    labels = torch.tensor([item['label'] for item in batch]) # получите список классов в батче
    lengths = torch.tensor([len(seq) for seq in texts]) # посчитайте список длин текстов в батче 
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0) # реализуйте пэддинг для текстов


    return {
        'input_ids': padded_texts, 
        'lengths': lengths, 
        'labels': labels
    }


# создаём датасеты
train_dataset = YelpDataset(texts=train_texts_tokenized, labels=train_labels.tolist())
val_dataset = YelpDataset(texts=val_texts_tokenized, labels=val_labels.tolist())


batch_size = 64


# создаём даталоадеры
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


print(f'Количество батчей в train_dataloader: {len(train_dataloader)}')
print(f'Количество батчей в val_dataloader: {len(val_dataloader)}')


print('Размерности батчей:')
for batch in train_dataloader:
    print('input_ids:', batch['input_ids'].shape)
    print('lengths:', batch['lengths'].shape)
    print('labels:', batch['labels'].shape)
    break